#### Configurações iniciais do projeto

In [ ]:
import urllib.request
import urllib.error
import re
import time
import datetime
import os
import pandas as pd
from bs4 import BeautifulSoup
from sqlalchemy import create_engine, text

os.makedirs("database", exist_ok=True)

ENGINE = create_engine("sqlite:///database/linguagens.db", echo=True)
INICIAIS_PERMITIDAS = set("CJP")
MAX_LINGUAGENS = 40
URL_INDICE = "https://en.wikipedia.org/wiki/List_of_programming_languages"
HEADERS = {
    "User-Agent": (
        "Mozilla/5.0 (compatible; ProjetoInfnet/1.0; " "+https://www.infnet.edu.br)"
    )
}

# Rótulos identificados durante a análise exploratória das infoboxes 
# das linguagens da Wikipédia.

CAMPOS = {
    "ano_criacao": {
        "appeared",
        "created",
        "designed",
        "first appeared",
        "initial release",
        "release",
        "founded",
        "formation",
        "year",
    },
    "criador": {
        "author",
        "creator",
        "designer",
        "designed by",
        "original author",
        "original authors",
        "writer",
    },
    "paradigma": {"paradigm", "paradigms"},
    "tipagem": {"typing", "type system", "typing discipline"},
}

#### Criação de Tabelas

In [ ]:
def criar_tabelas(engine):
    with engine.connect() as conn:
        conn.execute(text("""
                          CREATE TABLE IF NOT EXISTS linguagens 
                          (
                              id INTEGER PRIMARY KEY,
                              nome TEXT,
                              ano_criacao INTEGER,
                              criador TEXT,
                              paradigma TEXT,
                              tipagem TEXT,
                              url_wikipedia TEXT
                          )
                                                    """))
        conn.execute(text("""
                          CREATE TABLE IF NOT EXISTS paginas_visitadas 
                          (
                              id INTEGER PRIMARY KEY,
                              url TEXT,
                              http_status INTEGER,
                              timestamp TEXT,
                              linguagem_id INTEGER,
                              FOREIGN KEY (linguagem_id) REFERENCES linguagens(id)
                          )
                          """))
        conn.execute(text("""
                          CREATE TABLE IF NOT EXISTS erros_scraping 
                          (
                            id INTEGER PRIMARY KEY,
                            url TEXT,
                            tipo_erro TEXT,
                            mensagem TEXT,
                            timestamp TEXT      
                          )
                          
                          """))
        conn.commit()
        
        
criar_tabelas(ENGINE)

In [ ]:
def analisar_rotulos(links: list[tuple[str, str]]):

    rotulos_encontrados = set()

    for nome, url in links:

        status, html = download_pagina(url)
        soup = BeautifulSoup(html, "html.parser")

        for linha in soup.select("tr"):

            th = linha.find("th", class_="infobox-label")

            if th is None:
                continue

            th = th.get_text(separator=" ", strip=True).lower()
            th = th.replace("\xa0", " ")

            rotulos_encontrados.add(th)

    for rotulo in sorted(rotulos_encontrados):
        print(rotulo)

#### Download da página índice

In [ ]:
def download_pagina(url: str):
    req = urllib.request.Request(url, headers=HEADERS)
    with urllib.request.urlopen(req, timeout=10) as resp:
        status = resp.status
        html = resp.read()
    return status, html
        
download_pagina(URL_INDICE)  

#### Extração dos links de cada linguagem

In [ ]:
def get_links_linguagens(url:str):
    
    status, html = download_pagina(url)
    
    soup = BeautifulSoup(html, 'html.parser') #converte string html em objeto Beautifulsoup
    
    links_linguagens = [] 
    for link in soup.select('div.div-col li a'):
       href = str(link.get('href', ''))
       nome = link.get_text(strip=True)
       if not nome or not href.startswith('/wiki/'):
            continue
       if nome[0].upper() not in INICIAIS_PERMITIDAS:
            continue
       url_linguagem = 'https://en.wikipedia.org' + str(href) 
       links_linguagens.append((nome, url_linguagem))
       
    return links_linguagens
        
get_links_linguagens(URL_INDICE)

#### Análise Exploratória dos rótulos de dados necessários


In [ ]:
def analisar_rotulos(links: list[tuple[str, str]]):

    rotulos_encontrados = set()

    for nome, url in links:

        status, html = download_pagina(url)
        soup = BeautifulSoup(html, "html.parser")

        for linha in soup.select("tr"):

            th = linha.find("th", class_="infobox-label")

            if th is None:
                continue

            th = th.get_text(separator=" ", strip=True).lower()
            th = th.replace("\xa0", " ")

            rotulos_encontrados.add(th)

    for rotulo in sorted(rotulos_encontrados):
        print(rotulo)
        
analisar_rotulos(get_links_linguagens(URL_INDICE))

#### Captura, gestão de erros e salvamento dos dados

In [ ]:
def get_dados(links: list[tuple[str, str]]):

    for nome, url in links:
        try:
            status, html = download_pagina(url)
            if status != 200:
                continue
            print(f"Status HTTP: {status}. Iniciando scraping...")

            soup = BeautifulSoup(html, "html.parser")
            ano_criacao = None
            criador = None
            paradigma = None
            tipagem = None

            # nome da linguagem
            nome = soup.find("h1", id="firstHeading")
            if nome:
                nome = nome.get_text(strip=True)
            else:
                "desconhecido"

            # acessando table headers e obtendo identificador do CAMPO
            for linha in soup.select("tr"):
                th = linha.find("th", class_="infobox-label")

                if th is None:
                    continue

                th = th.get_text(separator=" ", strip=True).lower()
                th = th.replace("\xa0", " ")

                # ano_criacao
                if th in CAMPOS["ano_criacao"]:
                    td = linha.find("td", class_="infobox-data")

                    if td is not None:
                        ano = td.get_text(" ", strip=True)
                        ano = re.search(r"\b(\d{4})\b", ano)

                        if ano is not None:
                            ano_criacao = int(ano.group(1))

                # criador
                if th in CAMPOS["criador"]:
                    td = linha.find("td", class_="infobox-data")

                    if td is not None:
                        criador = td.get_text(" ", strip=True)

                # paradigma
                if th in CAMPOS["paradigma"]:
                    td = linha.find("td", class_="infobox-data")

                    if td is not None:
                        paradigma = td.get_text(" ", strip=True)

                # tipagem
                if th in CAMPOS["tipagem"]:
                    td = linha.find("td", class_="infobox-data")

                    if td is not None:
                        tipagem = td.get_text(" ", strip=True)

            print(nome, ano_criacao, criador, paradigma, tipagem)

            """ Codigo para salvar a linguagem e os dados. Nao foi necessário criar ou receber id pois esse atributo é autoincrementavel na tabela do SQLLite. É boa prática deixar o banco gerar os ids."""

            df_linguagem = pd.DataFrame(
                [
                    {
                        "nome": nome,
                        "ano_criacao": ano_criacao,
                        "criador": criador,
                        "paradigma": paradigma,
                        "tipagem": tipagem,
                        "url_wikipedia": url,
                    }
                ]
            )

            df_linguagem.to_sql("linguagens", ENGINE, if_exists="append", index=False)

            df_id = pd.read_sql(
                """
                    SELECT id
                    FROM linguagens
                    WHERE url_wikipedia = ?
                    """,
                ENGINE,
                params=(url,),
            )

            linguagem_id = int(df_id.iloc[0]["id"])

            # Codigo para salvar a visita

            df_paginas_visitadas = pd.DataFrame(
                [
                    {
                        "url": url,
                        "http_status": status,
                        "timestamp": datetime.datetime.now().strftime(
                            "%Y-%m-%d %H:%M:%S"
                        ),
                        "linguagem_id": linguagem_id,
                    }
                ]
            )

            df_paginas_visitadas.to_sql(
                "paginas_visitadas", ENGINE, if_exists="append", index=False
            )
            
        #erros de resposta do servidor
        except urllib.error.HTTPError as e: 
            df_erro = pd.DataFrame(
                [
                    {
                        "url": url,
                        "tipo_erro": type(e).__name__,
                        "mensagem": str(e),
                        "timestamp": datetime.datetime.now().strftime(
                            "%Y-%m-%d %H:%M:%S"
                        ),
                    }
                ]
            )
            df_erro.to_sql("erros_scraping", ENGINE, if_exists="append", index=False)
            pass
        
        #erros de conexão ou acesso à URL
        except urllib.error.URLError as e:            
            df_erro = pd.DataFrame(
                [
                    {
                        "url": url,
                        "tipo_erro": type(e).__name__,
                        "mensagem": str(e),
                        "timestamp": datetime.datetime.now().strftime(
                            "%Y-%m-%d %H:%M:%S"
                        ),
                    }
                ]
            )
            df_erro.to_sql("erros_scraping", ENGINE, if_exists="append", index=False)
            pass
    
        # Não foi implementado tratamento para AttributeError, 
        # pois os campos são inicializados com None e permanecem 
        # assim quando a informação não está disponível na infobox.       
        
        # Mantive o Exception a seguir para erros inesperados 
        # não contemplados pelos tratamentos acima.
        
        except Exception as e:
            df_erro = pd.DataFrame(
                [
                    {
                        "url": url,
                        "tipo_erro": type(e).__name__,
                        "mensagem": str(e),
                        "timestamp": datetime.datetime.now().strftime(
                            "%Y-%m-%d %H:%M:%S"
                        ),
                    }
                ]
            )
            df_erro.to_sql("erros_scraping", ENGINE, if_exists="append", index=False)
            pass

        finally:
            time.sleep(0.5)


links = get_links_linguagens(URL_INDICE)

get_dados(links[:2])